Tool-Using Research Agent

An AI-powered research agent built with Python and Gemini. The agent uses external tools such as web search and a calculator to research questions, collect evidence, select relevant sources, and generate answers with supporting evidence.

Key Features
Gemini-powered reasoning
Web search and calculator tools
Evidence collection
Source selection
Claim-level traceability
Duplicate-search protection
Tool and API failure handling
Hard step limit for controlled execution

The project was developed and tested using Jupyter Notebook and Python.

In [121]:
result = research(
    "What is the capital of Japan?"
)

display_fallback_result(result)

Starting research...
Question: What is the capital of Japan?
Starting source-based research...
Question: What is the capital of Japan?

Web search completed.
Number of results: 5
Best sources selected: 3

Source-based research completed successfully.
FINAL ANSWER:
The capital of Japan is Tokyo. [Source 1]

ANSWER TRACEABILITY:
Traceable: True
Support score: 1.0
Reason: Important words from the answer were found in the supporting sources.

SUPPORTING SOURCES:

1. Capital of Japan - Wikipedia
   URL: https://en.wikipedia.org/wiki/Capital_of_Japan

2. Tokyo - Wikipedia
   URL: https://en.wikipedia.org/wiki/Tokyo

3. Capital of Japan - Simple English Wikipedia, the free ...
   URL: https://simple.wikipedia.org/wiki/Capital_of_Japan


In [13]:
# Install the official Google GenAI Python SDK
!pip install -U google-genai python-dotenv

In [ ]:
# Import os to access environment variables
import os

# Import load_dotenv to load our .env file
from dotenv import load_dotenv

# Import the Google GenAI client
from google import genai

# Load the .env file
load_dotenv()

# Read the Gemini API key
api_key = os.getenv("GEMINI_API_KEY")

# Create the Gemini client
client = genai.Client(api_key=api_key)

# Ask Gemini a simple question
response = client.models.generate_content(
    model="gemini-3.5-flash",
    contents="What is an AI research agent?"
)

# Display the response
print(response.text)

In [17]:
# Install DuckDuckGo search package
!pip install -U ddgs


In [70]:
# Import the DuckDuckGo search class
from ddgs import DDGS

# Create a search function
def web_search(query):
    """
    Search the web using DuckDuckGo.

    Args:
        query: The research question or search keywords.

    Returns:
        A list of search results.
    """

    # Create a DuckDuckGo search client
    search_client = DDGS()

    # Perform the web search
    results = search_client.text(
        query,
        max_results=5
    )

    # Return the search results
    return results

In [71]:
# Test our web search tool
results = web_search("What is artificial intelligence?")

# Display the results
for result in results:
    print("Title:", result.get("title"))
    print("URL:", result.get("href"))
    print("Description:", result.get("body"))
    print("-" * 60)

Title: Artificial intelligence - Wikipedia
URL: https://en.wikipedia.org/wiki/Artificial_intelligence
Description: Artificial intelligence (AI) is the capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and decision-making.
------------------------------------------------------------
Title: What is artificial intelligence (AI)? - IBM
URL: https://www.ibm.com/think/topics/artificial-intelligence
Description: Artificial intelligence (AI) is technology that enables computers and machines to simulate human learning, comprehension, problem solving, decision-making, creativity and autonomy.
------------------------------------------------------------
Title: Artificial intelligence (AI) | Definition, Examples, Types ...
URL: https://www.britannica.com/technology/artificial-intelligence
Description: Artificial intelligence (AI) is the ability of a digital computer or computer-controlled rob

In [72]:
#Build the Webpage Fetch Tool
# Install the libraries needed to fetch and read webpages. Install BeautifulSoup
!pip install -U requests beautifulsoup4

In [21]:
!pip install python-dotenv

In [22]:
import os

print(os.getcwd())

C:\Users\USER\Desktop\tool-using-research-agent


In [23]:
def calculator(expression):
    try:
        result = eval(expression, {"__builtins__": {}}, {})
        return str(result)
    except Exception as e:
        return f"Calculator error: {e}"

In [24]:
result = calculator("125 * 48")

print(result)

6000


In [25]:
print(calculator("250 + 150"))

400


In [26]:
evidence = []

In [73]:
result = calculator("125 * 48")

evidence.append({
    "tool": "calculator",
    "input": "125 * 48",
    "output": result
})

print(evidence)

[{'tool': 'calculator', 'input': '125 * 48', 'output': '6000'}]


In [69]:
tool_description = """
You are a research agent.

You have access to this tool:

CALCULATOR
- Use it when the user asks for a mathematical calculation.
- Input must be a mathematical expression.
- The calculator returns the result.

If a question does not require calculation, answer normally.
"""

question = "What is 125 multiplied by 48?"

response = client.models.generate_content(
    model="gemini-3.5-flash",
    contents=tool_description + "\n\nUser question: " + question
)

print(response.text)

Next step

We'll use Gemini's native function/tool calling so Gemini can actually decide:

“I need the calculator → call the calculator → get the result → use that result in my answer.”

That is the point where it starts becoming a real tool-using agent, rather than just a Gemini chatbot.

In [54]:
# ---------------------------------------------------------
# TOOL 1: CALCULATOR
# ---------------------------------------------------------

def calculator(expression):
    """
    Calculate a mathematical expression.
    """

    try:
        # Evaluate the mathematical expression
        result = eval(
            expression,
            {"__builtins__": {}},
            {}
        )

        # Return the result as text
        return str(result)

    except Exception as e:
        # Return an error instead of crashing the program
        return f"Calculator error: {e}"


print("Calculator tool is ready.")

Calculator tool is ready.


In [30]:
# ---------------------------------------------------------
# TEST THE CALCULATOR
# ---------------------------------------------------------

# Give a mathematical expression to our calculator
result = calculator("125 * 48")

# Display the result
print(result)

6000


In [31]:
# Test another mathematical calculation
result = calculator("(250 + 150) / 4")

# Display the result
print(result)

100.0


In [57]:
# ---------------------------------------------------------
# DEFINE THE CALCULATOR TOOL FOR GEMINI
# ---------------------------------------------------------

# This description tells Gemini what the calculator does.
# Gemini can use this information to decide when the tool
# is useful.
calculator_tool = {
    
    # Name of our tool
    "name": "calculator",
    
    # Explain the purpose of the tool
    "description": "Calculate a mathematical expression.",
    
    # Define what information Gemini must provide
    "parameters": {
        
        # We are expecting an object containing parameters
        "type": "object",
        
        # Define the parameter that the calculator needs
        "properties": {
            
            # The calculator needs a mathematical expression
            "expression": {
                "type": "string",
                "description": "The mathematical expression to calculate."
            }
        },
        
        # Tell Gemini that expression is required
        "required": ["expression"]
    }
}

In [59]:
# ---------------------------------------------------------
# GIVE THE CALCULATOR TOOL TO GEMINI
# ---------------------------------------------------------

# Ask Gemini a question that requires calculation
question = "Calculate 125 multiplied by 48."

# Send the question to Gemini along with our calculator tool
response = client.models.generate_content(
    
    # Use the Gemini model
    model="gemini-3.5-flash",
    
    # Send the user's question
    contents=question,
    
    # Tell Gemini that it has access to our calculator
    config={
        "tools": [
            {
                "function_declarations": [calculator_tool]
            }
        ]
    }
)

# Display Gemini's response
print(response)

sdk_http_response=HttpResponse(
  headers=<dict len=12>
) candidates=[Candidate(
  content=Content(
    parts=[
      Part(
        function_call=FunctionCall(
          args={
            'expression': '125 * 48'
          },
          id='call_7026591',
          name='calculator'
        ),
        thought_signature=b'\x12\xf2\x01\n\xef\x01\x01\x11M2\x0fH\xcf\x84\xfe$\xe3\x96\x89\no\xa1o\xd28l~\xe1eC\x860\x8d\xde\xe9\xf0\x8b\xd2\xe3-\xc3C\x7f0\xb2`\xfa\x80\x7f\xe7\x19F14,.\x80\xc6\x99R\x93\xf3\x10A\xa0\x10\xe9\xdf\xe6\x90\xdf\x8ef\xbai\xa1r<\xb9l\xa68x\xff,\x7f!\xf6\x85\xb6\xb8\xee\x820\x7f\xd9v...'
      ),
    ],
    role='model'
  ),
  finish_reason=<FinishReason.STOP: 'STOP'>,
  index=0
)] create_time=None model_version='gemini-3.5-flash' prompt_feedback=None response_id='C_CoapbQA-WjqfkP6YOQ8QI' usage_metadata=GenerateContentResponseUsageMetadata(
  candidates_token_count=20,
  prompt_token_count=63,
  prompt_tokens_details=[
    ModalityTokenCount(
      modality=<MediaModalit

In [60]:
# ---------------------------------------------------------
# HANDLE GEMINI'S TOOL CALL
# ---------------------------------------------------------

# Check whether Gemini requested a function/tool call
if response.function_calls:
    
    # Get the first function call requested by Gemini
    function_call = response.function_calls[0]
    
    # Display the name of the requested tool
    print("Tool requested:", function_call.name)
    
    # Display the arguments Gemini provided
    print("Arguments:", function_call.args)
    
else:
    # Gemini did not request a tool
    print("Gemini did not request a tool.")

Tool requested: calculator
Arguments: {'expression': '125 * 48'}


In [61]:
# ---------------------------------------------------------
# RUN THE TOOL REQUESTED BY GEMINI
# ---------------------------------------------------------

# Check whether Gemini requested a tool
if response.function_calls:

    # Get the first function call
    function_call = response.function_calls[0]

    # Get the name of the requested tool
    tool_name = function_call.name

    # Get the arguments provided by Gemini
    tool_args = function_call.args

    # Display what Gemini requested
    print("Tool requested:", tool_name)
    print("Arguments:", tool_args)

    # -----------------------------------------------------
    # EXECUTE THE CALCULATOR TOOL
    # -----------------------------------------------------

    # Check if Gemini requested our calculator
    if tool_name == "calculator":

        # Get the mathematical expression
        expression = tool_args["expression"]

        # Run our calculator function
        tool_result = calculator(expression)

        # Display the result
        print("Calculator result:", tool_result)

    else:

        # Handle an unknown tool
        tool_result = "Unknown tool requested."

        print(tool_result)

else:

    # Gemini did not request a tool
    print("No tool was requested.")

Tool requested: calculator
Arguments: {'expression': '125 * 48'}
Calculator result: 6000


In [62]:
# ---------------------------------------------------------
# SEND THE TOOL RESULT BACK TO GEMINI
# ---------------------------------------------------------

# Import the Google GenAI types
from google.genai import types

# Create a function-response part.
# This tells Gemini what our calculator returned.
tool_response = types.Part.from_function_response(
    
    # Tell Gemini which tool produced the result
    name=tool_name,
    
    # Put the calculator result inside the response
    response={
        "result": tool_result
    }
)

# Send the original question, Gemini's tool request,
# and our tool result back to Gemini.
final_response = client.models.generate_content(
    
    # Use the same Gemini model
    model="gemini-3.5-flash",
    
    # Give Gemini the conversation so far
    contents=[
        
        # The original user question
        question,
        
        # Gemini's previous response containing the tool call
        response.candidates[0].content,
        
        # The result returned by our calculator
        tool_response
    ],
    
    # Give Gemini access to the calculator tool again
    config={
        "tools": [
            {
                "function_declarations": [calculator_tool]
            }
        ]
    }
)

# Display Gemini's final answer
print(final_response.text)

125 multiplied by 48 is 6,000.


In [63]:
# ---------------------------------------------------------
# INSTALL WEB SEARCH PACKAGE
# ---------------------------------------------------------

# Install the DDGS package.
# This package allows Python to perform web searches.
!pip install -U ddgs

In [64]:
# ---------------------------------------------------------
# TEST THE WEB SEARCH TOOL
# ---------------------------------------------------------

# Ask the web-search tool to search for something
results = web_search("What is artificial intelligence?")

# Display the results
print(results)

[{'title': 'Artificial intelligence - Wikipedia', 'href': 'https://en.wikipedia.org/wiki/Artificial_intelligence', 'body': '13 hours ago - Artificial intelligence (AI) is the capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and decision-making.'}, {'title': 'What Is Artificial Intelligence (AI)? | IBM', 'href': 'https://www.ibm.com/think/topics/artificial-intelligence', 'body': 'November 21, 2024 - Artificial intelligence (AI) is technology that enables computers and machines to simulate human learning, comprehension, problem solving, decision making, creativity and autonomy.'}, {'title': 'What is Artificial Intelligence (AI)? | Google Cloud', 'href': 'https://cloud.google.com/learn/what-is-artificial-intelligence', 'body': '13 hours ago - Artificial intelligence (AI) is a set of technologies that empowers computers to learn, reason, and perform a variety of advanced tasks in way

In [65]:
# ---------------------------------------------------------
# DEFINE THE WEB SEARCH TOOL FOR GEMINI
# ---------------------------------------------------------

# This description tells Gemini what the web search tool does.
web_search_tool = {
    
    # Name of our tool
    "name": "web_search",
    
    # Explain when Gemini should use this tool
    "description": (
        "Search the web for current or factual information "
        "and return relevant search results with sources."
    ),
    
    # Define the information Gemini needs to provide
    "parameters": {
        
        # The parameters are provided as an object
        "type": "object",
        
        # Define the query parameter
        "properties": {
            
            # The search query Gemini should create
            "query": {
                "type": "string",
                "description": "The question or topic to search for on the web."
            }
        },
        
        # Gemini must provide a search query
        "required": ["query"]
    }
}

# Display the tool definition
print("Web search tool is ready.")

Web search tool is ready.


In [66]:
# ---------------------------------------------------------
# GIVE BOTH TOOLS TO GEMINI
# ---------------------------------------------------------

# Create a list containing both of our tools
tools = [
    
    # Calculator tool
    calculator_tool,
    
    # Web search tool
    web_search_tool
]

# Ask Gemini a question that requires web research
question = "Who is the current CEO of Microsoft?"

# Send the question and both tools to Gemini
response = client.models.generate_content(
    
    # Use our Gemini model
    model="gemini-3.5-flash",
    
    # Send the user's question
    contents=question,
    
    # Give Gemini access to both tools
    config={
        "tools": [
            {
                "function_declarations": tools
            }
        ]
    }
)

# Check whether Gemini requested a tool
if response.function_calls:
    
    # Get the first requested tool
    function_call = response.function_calls[0]
    
    # Display the tool Gemini selected
    print("Tool requested:", function_call.name)
    
    # Display the arguments Gemini provided
    print("Arguments:", function_call.args)

else:
    
    # Gemini answered without using a tool
    print("Gemini did not request a tool.")

Tool requested: web_search
Arguments: {'query': 'current CEO of Microsoft'}


In [46]:
# ---------------------------------------------------------
# RUN THE WEB SEARCH TOOL REQUESTED BY GEMINI
# ---------------------------------------------------------

# Check whether Gemini requested a tool
if response.function_calls:

    # Get the first function call
    function_call = response.function_calls[0]

    # Get the name of the requested tool
    tool_name = function_call.name

    # Get the arguments Gemini provided
    tool_args = function_call.args

    # Display the requested tool
    print("Tool requested:", tool_name)

    # Display the arguments
    print("Arguments:", tool_args)

    # -----------------------------------------------------
    # EXECUTE THE WEB SEARCH
    # -----------------------------------------------------

    # Check whether Gemini requested web search
    if tool_name == "web_search":

        # Get the search query created by Gemini
        search_query = tool_args["query"]

        # Run our web search function
        search_results = web_search(search_query)
        

        # Display the search results
        print("\nWeb search results:")
        print(search_results)

    else:

        # Handle an unexpected tool
        print("Unexpected tool:", tool_name)

else:

    # Gemini did not request a tool
    print("No tool was requested.")
    

No tool was requested.


The next step is:

Web search results → Gemini → final answer

Then Gemini can say something like:

Satya Nadella is the current CEO of Microsoft.

Gemini + Web Search tool loop is working.

Now we have completed:

✅ Gemini
✅ Calculator tool
✅ Web Search tool
✅ Gemini choosing the correct tool
✅ Tool execution
✅ Sending the tool result back to Gemini
✅ Final answer generation
Next important part: Evidence Store

This is important because your project requirement says that final claims should be traceable to the information fetched by the tools.

In [68]:
# ---------------------------------------------------------
# HARD STEP LIMIT
# ---------------------------------------------------------

# Maximum number of tool calls the research agent can make
MAX_STEPS = 5

# Keep track of how many tool calls have been made
step_count = 0

print("Maximum tool calls allowed:", MAX_STEPS)
#This means later, when we build the Research Agent loop, it will never be allowed to make more than 5 tool calls for one research questio

Maximum tool calls allowed: 5


#Now we’ll add tool-failure handling. This is another project requirement: if a tool fails, the agent should not crash.

In [75]:
# ---------------------------------------------------------
# TOOL FAILURE HANDLING
# ---------------------------------------------------------

def run_tool_safely(tool_function, *args):
    """
    Run a tool safely.

    If the tool works, return its result.
    If the tool fails, return an error message instead
    of stopping the entire research agent.
    """

    try:
        # Run the requested tool
        result = tool_function(*args)

        # Return the successful result
        return {
            "success": True,
            "result": result
        }

    except Exception as e:
        # Catch any error from the tool
        return {
            "success": False,
            "error": str(e)
        }


print("Tool failure handling is ready.")

Tool failure handling is ready.


In [ ]:
Now we have all the basic building blocks. The next step is the most important one: combine them into one Research Agent.

In [51]:
# ---------------------------------------------------------
# IMPROVED RESEARCH AGENT WITH SOURCE CHECKING
# ---------------------------------------------------------

def research_agent_v2(question):
    """
    Research the user's question using Gemini and available tools.
    The agent also checks which web sources are relevant.
    """

    # Start the step counter
    step_count = 0

    # Store evidence collected during the research
    evidence = []

    # Ask Gemini which tool should be used
    response = client.models.generate_content(
        model="gemini-3.5-flash",
        contents=question,
        config={
            "tools": [
                {
                    "function_declarations": tools
                }
            ]
        }
    )

    # Continue while Gemini requests a tool
    while response.function_calls:

        # Stop requesting more tools when the limit is reached
        if step_count >= MAX_STEPS:

            print("\nMaximum step limit reached.")
            print("Using the evidence collected so far.")

            # Stop the tool loop
            break

        # Increase the step counter
        step_count += 1

        # Get the first requested tool
        function_call = response.function_calls[0]

        # Get the tool name
        tool_name = function_call.name

        # Get the tool arguments
        tool_args = function_call.args

        print("Step:", step_count)
        print("Tool requested:", tool_name)
        print("Arguments:", tool_args)

        # -------------------------------------------------
        # RUN WEB SEARCH
        # -------------------------------------------------

        if tool_name == "web_search":

            # Get the search query
            search_query = tool_args["query"]

            # Run web search safely
            tool_output = run_tool_safely(
                web_search,
                search_query
            )

        # -------------------------------------------------
        # RUN CALCULATOR
        # -------------------------------------------------

        elif tool_name == "calculator":

            # Get the mathematical expression
            expression = tool_args["expression"]

            # Run calculator safely
            tool_output = run_tool_safely(
                calculator,
                expression
            )

        # -------------------------------------------------
        # UNKNOWN TOOL
        # -------------------------------------------------

        else:

            tool_output = {
                "success": False,
                "error": f"Unknown tool: {tool_name}"
            }

        # Save the result as evidence
        evidence.append({
            "step": step_count,
            "tool": tool_name,
            "arguments": tool_args,
            "output": tool_output
        })

        # Stop if the tool failed
        if not tool_output["success"]:

            return {
                "success": False,
                "answer": f"The {tool_name} tool failed.",
                "evidence": evidence,
                "sources": []
            }

        # Send the tool result back to Gemini
        from google.genai import types

        tool_response = types.Part.from_function_response(
            name=tool_name,
            response={
                "result": tool_output["result"]
            }
        )

        # Ask Gemini to continue
        response = client.models.generate_content(
            model="gemini-3.5-flash",
            contents=[
                question,
                response.candidates[0].content,
                tool_response
            ],
            config={
                "tools": [
                    {
                        "function_declarations": tools
                    }
                ]
            }
        )

    # -----------------------------------------------------
    # CHECK WEB SOURCES
    # -----------------------------------------------------

    relevant_sources = []

    # Look through the collected evidence
    for item in evidence:

        # Only web-search results need source checking
        if item["tool"] == "web_search":

            search_results_from_agent = item["output"]["result"]

            # Ask Gemini which sources are relevant
            source_numbers = check_source_relevance(
                question,
                search_results_from_agent
            )

            print(
                "\nRelevant source numbers:",
                source_numbers
            )

            # Convert source numbers into integers
            try:

                numbers = [
                    int(number.strip())
                    for number in source_numbers.split(",")
                    if number.strip().isdigit()
                ]

                # Select the relevant sources
                for number in numbers:

                    if 1 <= number <= len(search_results_from_agent):

                        relevant_sources.append(
                            search_results_from_agent[number - 1]
                        )

            except Exception:

                # Keep the source list empty if checking fails
                pass

    # -----------------------------------------------------
    # CREATE FINAL RESULT
    # -----------------------------------------------------

    # If the step limit was reached, Gemini's latest answer
    # may not be available. We therefore provide a clear
    # message rather than pretending the research is complete.

    if step_count >= MAX_STEPS:

        final_answer = (
            "The research agent reached its maximum step limit. "
            "The evidence collected so far has been preserved."
        )

    else:

        final_answer = response.text

    # Return the complete research result
    return {
        "success": True,
        "answer": final_answer,
        "evidence": evidence,
        "sources": relevant_sources
    }


print("Improved Research Agent is ready.")

Improved Research Agent is ready.


Your Research Agent is working correctly.

Your test proves that:

✅ Gemini understood the question.
✅ It selected the Web Search tool.
✅ The web search ran successfully.
✅ Gemini used the search result to produce the answer.
✅ The agent recorded 1 evidence item.
✅ The hard step limit is active.
✅ Tool-failure handling is included.
Next step: make the answer traceable

Right now, your agent gives:

The current CEO of Microsoft is Satya Nadella.

But the project requirement says the final claims should be traceable to the fetched source.

In [ ]:
# ---------------------------------------------------------
# TEST THE RESEARCH AGENT WITH THE CALCULATOR
# ---------------------------------------------------------

# Ask the research agent a mathematical question
result = research_agent(
    "What is 125 multiplied by 48?"
)

# Display the final answer
print("FINAL ANSWER:")
print(result["answer"])

# Display the evidence collected
print("\nEVIDENCE COLLECTED:")
print(len(result["evidence"]))

In [ ]:
# ---------------------------------------------------------
# TEST TOOL FAILURE HANDLING
# ---------------------------------------------------------

# Ask the research agent to calculate an invalid expression
result = research_agent(
    "Calculate this: 10 / 0"
)

# Display the result
print("FINAL ANSWER:")
print(result["answer"])

# Display whether the research was successful
print("\nSUCCESS:")
print(result["success"])

# Display the evidence collected
print("\nEVIDENCE COLLECTED:")
print(len(result["evidence"]))

In [ ]:
# ---------------------------------------------------------
# DIRECT TEST OF TOOL FAILURE HANDLING
# ---------------------------------------------------------

# Give the calculator an invalid mathematical expression
test = run_tool_safely(
    calculator,
    "10 / 0"
)

# Display the result
print("SUCCESS:")
print(test["success"])

print("\nRESULT:")
print(test)

In [ ]:
# ---------------------------------------------------------
# FILTER RELEVANT WEB SOURCES
# ---------------------------------------------------------

def filter_sources(search_results, question):
    """
    Keep search results that are more closely related
    to the user's question.
    """

    # Convert the question into lowercase words
    question_words = set(question.lower().split())

    # Create an empty list for relevant sources
    relevant_sources = []

    # Check each search result
    for source in search_results:

        # Combine the title and description
        text = (
            source.get("title", "") + " " +
            source.get("body", "")
        ).lower()

        # Count how many question words appear in the result
        matches = sum(
            1 for word in question_words
            if word in text
        )

        # Keep the source if it has at least one match
        if matches > 0:
            relevant_sources.append(source)

    # Return the filtered sources
    return relevant_sources


print("Source filtering is ready.")

In [ ]:
# ---------------------------------------------------------
# TEST SOURCE FILTERING
# ---------------------------------------------------------

# Filter the web-search results from our earlier test
relevant_sources = filter_sources(
    search_results,
    question
)

# Display the relevant sources
print("RELEVANT SOURCES:")

for number, source in enumerate(relevant_sources, start=1):

    print(f"\n{number}. {source['title']}")
    print(f"   {source['href']}")

In [ ]:
Next step: let Gemini identify supporting sources

We'll add a small function that takes the search results and asks Gemini to select the sources that actually support the question.

In [ ]:
# ---------------------------------------------------------
# AI-BASED SOURCE RELEVANCE CHECK
# ---------------------------------------------------------

def check_source_relevance(question, search_results):
    """
    Ask Gemini to identify which search results
    are relevant to the research question.
    """

    # Convert the search results into readable text
    sources_text = ""

    for number, source in enumerate(search_results, start=1):

        sources_text += (
            f"\nSOURCE {number}\n"
            f"Title: {source.get('title', '')}\n"
            f"URL: {source.get('href', '')}\n"
            f"Description: {source.get('body', '')}\n"
        )

    # Create instructions for Gemini
    prompt = f"""
You are checking research sources.

Research question:
{question}

Below are web search results:

{sources_text}

Identify the source numbers that directly support the answer
to the research question.

Return ONLY the relevant source numbers as a comma-separated list.
For example:
1,3,5
"""

    # Ask Gemini to evaluate the sources
    response = client.models.generate_content(
        model="gemini-3.5-flash",
        contents=prompt
    )

    # Return Gemini's answer
    return response.text.strip()


print("AI source relevance checker is ready.")

In [ ]:
# ---------------------------------------------------------
# TEST AI SOURCE RELEVANCE CHECK
# ---------------------------------------------------------

# Ask Gemini which sources support our question
relevant_source_numbers = check_source_relevance(
    question,
    search_results
)

# Display the result
print("RELEVANT SOURCE NUMBERS:")
print(relevant_source_numbers)

In [ ]:
Next step — integrate source checking

We will make a new version of the research_agent() function that:

Searches the web.
Stores the evidence.
Lets Gemini answer.
Checks which sources actually support the research.
Shows only the supporting sources.

In [ ]:
# ---------------------------------------------------------
# IMPROVED RESEARCH AGENT WITH SOURCE CHECKING
# ---------------------------------------------------------

def research_agent_v2(question):
    """
    Research the user's question using Gemini and available tools.
    The agent also checks which web sources are relevant.
    """

    # Start the step counter
    step_count = 0

    # Store evidence collected during the research
    evidence = []

    # Ask Gemini which tool should be used
    response = client.models.generate_content(
        model="gemini-3.5-flash",
        contents=question,
        config={
            "tools": [
                {
                    "function_declarations": tools
                }
            ]
        }
    )

    # Continue while Gemini requests a tool
    while response.function_calls:

        # Stop if the maximum number of steps is reached
        if step_count >= MAX_STEPS:
            return {
                "success": False,
                "answer": "Research stopped because the maximum step limit was reached.",
                "evidence": evidence,
                "sources": []
            }

        # Increase the step counter
        step_count += 1

        # Get the first requested tool
        function_call = response.function_calls[0]

        # Get the tool name
        tool_name = function_call.name

        # Get the tool arguments
        tool_args = function_call.args

        print("Step:", step_count)
        print("Tool requested:", tool_name)
        print("Arguments:", tool_args)

        # -------------------------------------------------
        # RUN WEB SEARCH
        # -------------------------------------------------

        if tool_name == "web_search":

            # Get the search query
            search_query = tool_args["query"]

            # Run web search safely
            tool_output = run_tool_safely(
                web_search,
                search_query
            )

        # -------------------------------------------------
        # RUN CALCULATOR
        # -------------------------------------------------

        elif tool_name == "calculator":

            # Get the mathematical expression
            expression = tool_args["expression"]

            # Run calculator safely
            tool_output = run_tool_safely(
                calculator,
                expression
            )

        # -------------------------------------------------
        # UNKNOWN TOOL
        # -------------------------------------------------

        else:

            # Handle an unknown tool
            tool_output = {
                "success": False,
                "error": f"Unknown tool: {tool_name}"
            }

        # Save the result as evidence
        evidence.append({
            "step": step_count,
            "tool": tool_name,
            "arguments": tool_args,
            "output": tool_output
        })

        # Stop if the tool failed
        if not tool_output["success"]:
            return {
                "success": False,
                "answer": f"The {tool_name} tool failed.",
                "evidence": evidence,
                "sources": []
            }

        # Send the tool result back to Gemini
        from google.genai import types

        tool_response = types.Part.from_function_response(
            name=tool_name,
            response={
                "result": tool_output["result"]
            }
        )

        # Ask Gemini to continue
        response = client.models.generate_content(
            model="gemini-3.5-flash",
            contents=[
                question,
                response.candidates[0].content,
                tool_response
            ],
            config={
                "tools": [
                    {
                        "function_declarations": tools
                    }
                ]
            }
        )

    # -----------------------------------------------------
    # CHECK WEB SOURCES
    # -----------------------------------------------------

    relevant_sources = []

    # Check whether web search was used
    for item in evidence:

        if item["tool"] == "web_search":

            # Get the search results
            search_results_from_agent = item["output"]["result"]

            # Ask Gemini which sources support the question
            source_numbers = check_source_relevance(
                question,
                search_results_from_agent
            )

            print("\nRelevant source numbers:", source_numbers)

            # Convert the returned numbers into a list
            try:
                numbers = [
                    int(number.strip())
                    for number in source_numbers.split(",")
                    if number.strip().isdigit()
                ]

                # Select only the relevant sources
                for number in numbers:

                    if 1 <= number <= len(search_results_from_agent):

                        relevant_sources.append(
                            search_results_from_agent[number - 1]
                        )

            except Exception:
                # If source checking fails, keep the source list empty
                relevant_sources = []

    # Return the final research result
    return {
        "success": True,
        "answer": response.text,
        "evidence": evidence,
        "sources": relevant_sources
    }


print("Improved Research Agent is ready.")

In [ ]:
# ---------------------------------------------------------
# TEST IMPROVED RESEARCH AGENT
# ---------------------------------------------------------

result_v2 = research_agent_v2(
    "Who is the current CEO of Microsoft?"
)

# Display the final answer
print("\nFINAL ANSWER:")
print(result_v2["answer"])

# Display the supporting sources
print("\nSUPPORTING SOURCES:")

for number, source in enumerate(
    result_v2["sources"],
    start=1
):

    print(f"\n{number}. {source['title']}")
    print(f"   {source['href']}")

In [ ]:
# ---------------------------------------------------------
# TEST V2 WITH CALCULATOR  (test the calculator with research_agent_v2)
# ---------------------------------------------------------

result_v2_calc = research_agent_v2(
    "What is 125 multiplied by 48?"
)

# Display the final answer
print("\nFINAL ANSWER:")
print(result_v2_calc["answer"])

# Display the evidence collected
print("\nEVIDENCE COLLECTED:")

for item in result_v2_calc["evidence"]:
    print("\nStep:", item["step"])
    print("Tool:", item["tool"])
    print("Arguments:", item["arguments"])
    print("Output:", item["output"])

In [ ]:
# ---------------------------------------------------------
# TEST TOOL FAILURE HANDLING
# ---------------------------------------------------------

# Directly test the calculator with an invalid calculation.
# Division by zero should not crash the program.

failure_test = run_tool_safely(
    calculator,
    "10 / 0"
)

# Display whether the tool execution itself completed
print("TOOL EXECUTION COMPLETED:")
print(failure_test["success"])

# Display the actual result returned by the calculator
print("\nTOOL RESULT:")
print(failure_test["result"])


In [ ]:
# ---------------------------------------------------------
# FORMAT SUPPORTING SOURCES
# ---------------------------------------------------------

def format_sources(sources):
    """
    Convert the selected web sources into a simple
    numbered source list.
    """

    # Create an empty list to store formatted sources
    formatted_sources = []

    # Go through each selected source
    for number, source in enumerate(sources, start=1):

        # Get the source title
        title = source.get("title", "Unknown source")

        # Get the source URL
        url = source.get("href", "")

        # Create a numbered source entry
        formatted_sources.append(
            f"[Source {number}] {title}\n{url}"
        )

    # Join all sources into one text block
    return "\n\n".join(formatted_sources)


print("Source formatter is ready.")

In [ ]:
# ---------------------------------------------------------
# TEST SOURCE FORMATTER
# ---------------------------------------------------------

formatted = format_sources(
    result_v2["sources"]
)

print("SUPPORTING SOURCES:")
print(formatted)

In [ ]:
# ---------------------------------------------------------
# FINAL RESEARCH OUTPUT FORMATTER
# ---------------------------------------------------------

def display_research_result(result):
    """
    Display the research answer together with
    its supporting sources.
    """

    # Display the final answer
    print("FINAL ANSWER:")
    print(result["answer"])

    # Check whether supporting sources exist
    if result["sources"]:

        print("\nSUPPORTING SOURCES:")

        # Format and display the sources
        print(format_sources(result["sources"]))

    else:

        # Display this when no web sources were collected
        print("\nNO WEB SOURCES WERE USED.")


print("Final output formatter is ready.")

In [ ]:
# ---------------------------------------------------------
# TEST FINAL OUTPUT
# ---------------------------------------------------------

display_research_result(result_v2)


In [ ]:
# ---------------------------------------------------------
# TEST COMPLETE RESEARCH AGENT
# ---------------------------------------------------------

result_test = research_agent_v2(
    "What is the latest version of Python?"
)

# Display the final answer and supporting sources
display_research_result(result_test)

Question → Gemini → Web Search → Evidence → Source checking → Final answer → Sources

In [ ]:
# Test the research agent with a current-information question

result_test = research_agent_v2(
    "What is the latest stable version of Python?"
)

display_research_result(result_test)

In [91]:
# ---------------------------------------------------------
# GEMINI API FAILURE HANDLING
# ---------------------------------------------------------

def generate_gemini_response(question, use_tools=True):
    """
    Send a request to Gemini safely.

    If the Gemini API fails, return a clear error
    instead of stopping the entire research agent.
    """

    try:

        # Prepare the Gemini configuration
        config = {}

        # Add our tools when requested
        if use_tools:
            config = {
                "tools": [
                    {
                        "function_declarations": tools
                    }
                ]
            }

        # Send the request to Gemini
        response = client.models.generate_content(
            model="gemini-3.5-flash",
            contents=question,
            config=config
        )

        # Return successful response
        return {
            "success": True,
            "response": response
        }

    except Exception as e:

        # Return the error instead of crashing
        return {
            "success": False,
            "error": str(e)
        }


print("Gemini API failure handling is ready.")

Gemini API failure handling is ready.


In [93]:
# ---------------------------------------------------------
# WEB SEARCH FALLBACK WITH CLAIM-LEVEL TRACEABILITY
# ---------------------------------------------------------

def web_search_fallback(question):
    """
    Research a question directly using web search.

    This version:
    1. Searches the web.
    2. Selects the best sources.
    3. Detects conflicts.
    4. Resolves conflicts.
    5. Creates an answer.
    6. Checks whether the answer is supported by the sources.
    """

    print("Gemini is unavailable.")
    print("Using web search fallback...")

    # -----------------------------------------------------
    # STEP 1: SEARCH THE WEB
    # -----------------------------------------------------

    search_result = run_tool_safely(
        web_search,
        question
    )

    # -----------------------------------------------------
    # CHECK WHETHER SEARCH WORKED
    # -----------------------------------------------------

    if not search_result["success"]:

        return {
            "success": False,
            "answer": "Web search also failed.",
            "evidence": [],
            "sources": [],
            "traceability": {
                "traceable": False,
                "reason": "Web search failed."
            }
        }

    results = search_result["result"]

    # -----------------------------------------------------
    # STEP 2: SELECT THE BEST SOURCES
    # -----------------------------------------------------

    best_sources = select_best_sources(
        results
    )

    # -----------------------------------------------------
    # STEP 3: CHECK FOR SOURCE CONFLICTS
    # -----------------------------------------------------

    conflict_result = detect_source_conflicts(
        best_sources
    )

    print("\nCONFLICT DETECTED:")
    print(conflict_result["conflict"])

    # -----------------------------------------------------
    # STEP 4: HANDLE CONFLICTS
    # -----------------------------------------------------

    if conflict_result["conflict"]:

        resolution = resolve_source_conflict(
            best_sources
        )

        print("\nCONFLICT RESOLUTION:")
        print(resolution["answer"])

        answer = resolution["answer"]

    else:

        answer = create_fallback_answer(
            question,
            best_sources
        )

    # -----------------------------------------------------
    # STEP 5: CHECK CLAIM-LEVEL SUPPORT
    # -----------------------------------------------------

    claim_check = check_claim_support(
        answer,
        best_sources
    )

    print("\nCLAIM SUPPORT CHECK:")
    print("Supported:", claim_check["supported"])

    if "support_score" in claim_check:

        print(
            "Support score:",
            claim_check["support_score"]
        )

    print(
        "Reason:",
        claim_check["reason"]
    )

    # -----------------------------------------------------
    # STEP 6: CREATE EVIDENCE RECORD
    # -----------------------------------------------------

    evidence = [{
        "step": 1,
        "tool": "web_search",
        "arguments": {
            "query": question
        },
        "output": {
            "success": True,
            "result": best_sources
        }
    }]

    # -----------------------------------------------------
    # STEP 7: RETURN COMPLETE RESULT
    # -----------------------------------------------------

    return {
        "success": True,
        "answer": answer,
        "evidence": evidence,
        "sources": best_sources,
        "traceability": {
            "traceable": claim_check["supported"],
            "support_score": claim_check.get(
                "support_score",
                0
            ),
            "reason": claim_check["reason"]
        }
    }


print("Fallback with claim-level traceability is ready.")

Fallback with claim-level traceability is ready.


In [ ]:
# ---------------------------------------------------------
# TEST COMPLETE RESEARCH AGENT
# ---------------------------------------------------------

result = research_agent_v4(
    "What is the latest stable version of Python?"
)

display_fallback_result(result)

print("\nTRACEABILITY:")
print(result.get("traceability"))

In [ ]:
# ---------------------------------------------------------
# TEST WEB SEARCH FALLBACK
# ---------------------------------------------------------

fallback_test = web_search_fallback(
    "What is the latest stable version of Python?"
)

# Display the result
print("SUCCESS:")
print(fallback_test["success"])

print("\nANSWER:")
print(fallback_test["answer"])

print("\nNUMBER OF SOURCES:")
print(len(fallback_test["sources"]))

In [ ]:
# ---------------------------------------------------------
# TEST COMPLETE WEB SEARCH FALLBACK
# ---------------------------------------------------------

test_result = web_search_fallback(
    "What is the capital of Japan?"
)

# Display the complete result
display_fallback_result(
    test_result
)

In [94]:
# ---------------------------------------------------------
# DISPLAY FALLBACK RESULTS WITH TRACEABILITY
# ---------------------------------------------------------

def display_fallback_result(result):
    """
    Display the final answer, supporting sources,
    and answer traceability information.
    """

    # -----------------------------------------------------
    # DISPLAY FINAL ANSWER
    # -----------------------------------------------------

    print("FINAL ANSWER:")
    print(result["answer"])

    # -----------------------------------------------------
    # DISPLAY TRACEABILITY
    # -----------------------------------------------------

    traceability = result.get("traceability")

    if traceability:

        print("\nANSWER TRACEABILITY:")
        print("Traceable:", traceability["traceable"])

        if "support_score" in traceability:
            print(
                "Support score:",
                traceability["support_score"]
            )

        print(
            "Reason:",
            traceability["reason"]
        )

    # -----------------------------------------------------
    # DISPLAY SUPPORTING SOURCES
    # -----------------------------------------------------

    if result["sources"]:

        print("\nSUPPORTING SOURCES:")

        for number, source in enumerate(
            result["sources"],
            start=1
        ):

            print(
                f"\n{number}. "
                f"{source.get('title', 'Unknown title')}"
            )

            print(
                f"   URL: "
                f"{source.get('href', '')}"
            )

    else:

        print("\nNO SOURCES FOUND.")


print("Fallback result display with traceability is ready.")

Fallback result display with traceability is ready.


In [ ]:
# ---------------------------------------------------------
# TEST THE COMPLETE FALLBACK SYSTEM
# ---------------------------------------------------------

# Ask a research question
test_question = "What is the latest stable version of Python?"

# Run the web-search fallback
result = web_search_fallback(test_question)

# Display the complete result
display_fallback_result(result)

In [ ]:
# ---------------------------------------------------------
# DISPLAY THE PYTHON RESEARCH RESULTS
# ---------------------------------------------------------

display_fallback_result(fallback_test)

In [77]:
# ---------------------------------------------------------
# GENERAL SOURCE SELECTION
# ---------------------------------------------------------

def select_best_sources(search_results):
    """
    Select the best sources from web search results.

    More trustworthy sources are preferred, such as:
    - Official organization websites
    - Government websites
    - University websites
    - Well-known reference sources

    The function works for different topics instead of
    being limited to Python.org.
    """

    # -----------------------------------------------------
    # SOURCE AUTHORITY KEYWORDS
    # -----------------------------------------------------

    trusted_domains = [
        ".gov",
        ".edu",
        ".org"
    ]

    # -----------------------------------------------------
    # STORE SOURCES WITH THEIR SCORES
    # -----------------------------------------------------

    scored_sources = []

    for source in search_results:

        url = source.get(
            "href",
            ""
        ).lower()

        score = 0

        # -------------------------------------------------
        # GIVE HIGHER SCORE TO TRUSTED DOMAINS
        # -------------------------------------------------

        for domain in trusted_domains:

            if domain in url:
                score += 2

        # -------------------------------------------------
        # OFFICIAL MICROSOFT / GOOGLE / APPLE ETC.
        # -------------------------------------------------

        # Many official company websites use .com,
        # so we also look for common official domains.

        official_keywords = [
            "microsoft.com",
            "google.com",
            "apple.com",
            "python.org",
            "openai.com",
            "nasa.gov",
            "who.int"
        ]

        for official_domain in official_keywords:

            if official_domain in url:
                score += 3

        # -------------------------------------------------
        # SAVE SOURCE AND SCORE
        # -------------------------------------------------

        scored_sources.append(
            (score, source)
        )

    # -----------------------------------------------------
    # SORT BY TRUST SCORE
    # -----------------------------------------------------

    scored_sources.sort(
        key=lambda item: item[0],
        reverse=True
    )

    # -----------------------------------------------------
    # SELECT UP TO 3 SOURCES
    # -----------------------------------------------------

    selected_sources = [
        source
        for score, source in scored_sources[:3]
    ]

    return selected_sources


print("General source selection is ready.")

General source selection is ready.


In [ ]:
# ---------------------------------------------------------
# TEST BEST SOURCE SELECTION
# ---------------------------------------------------------

# Select the best sources from our previous search
best_sources = select_best_sources(
    fallback_test["sources"]
)

# Display the selected sources
print("BEST SOURCES:")

for number, source in enumerate(
    best_sources,
    start=1
):

    print(f"\n{number}. {source.get('title', 'Unknown title')}")
    print(f"   URL: {source.get('href', '')}")

In [ ]:
# ---------------------------------------------------------
# WEB SEARCH FALLBACK WITH CONFLICT RESOLUTION
# ---------------------------------------------------------

def web_search_fallback(question):
    """
    Research a question directly using web search.

    This version:
    1. Searches the web.
    2. Selects the best sources.
    3. Detects conflicting information.
    4. Resolves conflicts using authoritative sources.
    5. Creates the final answer.
    """

    print("Gemini is unavailable.")
    print("Using web search fallback...")

    # -----------------------------------------------------
    # RUN WEB SEARCH
    # -----------------------------------------------------

    search_result = run_tool_safely(
        web_search,
        question
    )

    # Check whether the web search worked
    if not search_result["success"]:

        return {
            "success": False,
            "answer": "Web search also failed.",
            "evidence": [],
            "sources": []
        }

    # Get all search results
    results = search_result["result"]

    # -----------------------------------------------------
    # SELECT THE BEST SOURCES
    # -----------------------------------------------------

    best_sources = select_best_sources(
        results
    )

    # -----------------------------------------------------
    # CHECK FOR SOURCE CONFLICTS
    # -----------------------------------------------------

    conflict_result = detect_source_conflicts(
        best_sources
    )

    print("\nCONFLICT DETECTED:")
    print(conflict_result["conflict"])

    # -----------------------------------------------------
    # RESOLVE CONFLICT IF FOUND
    # -----------------------------------------------------

    if conflict_result["conflict"]:

        resolution = resolve_source_conflict(
            best_sources
        )

        print("\nCONFLICT RESOLUTION:")
        print(resolution["answer"])

        answer = resolution["answer"]

    else:

        # No conflict, create the normal fallback answer
        answer = create_fallback_answer(
            question,
            best_sources
        )

    # -----------------------------------------------------
    # SAVE EVIDENCE
    # -----------------------------------------------------

    evidence = [{
        "step": 1,
        "tool": "web_search",
        "arguments": {
            "query": question
        },
        "output": {
            "success": True,
            "result": best_sources
        }
    }]

    # -----------------------------------------------------
    # RETURN FINAL RESULT
    # -----------------------------------------------------

    return {
        "success": True,
        "answer": answer,
        "evidence": evidence,
        "sources": best_sources
    }


print("Web search fallback with conflict resolution is ready.")

In [ ]:

# ---------------------------------------------------------
# TEST FALLBACK WITH CONFLICT RESOLUTION
# ---------------------------------------------------------

result = research_agent_v4(
    "What is the latest stable version of Python?"
)

display_fallback_result(result)

In [82]:
# ---------------------------------------------------------
# IMPROVED FALLBACK ANSWER GENERATOR - VERSION 2
# ---------------------------------------------------------

def create_fallback_answer(question, sources):
    """
    Create a short answer from web search results.

    The function checks the question type and tries to
    extract information that actually answers the question.
    """

    # -----------------------------------------------------
    # CHECK WHETHER SOURCES EXIST
    # -----------------------------------------------------

    if not sources:
        return (
            "No reliable sources were found, "
            "so an answer could not be created."
        )

    # -----------------------------------------------------
    # CLEAN THE QUESTION
    # -----------------------------------------------------

    question_lower = question.lower().strip()

    # -----------------------------------------------------
    # COMBINE SOURCE INFORMATION
    # -----------------------------------------------------

    source_text = ""

    for source in sources:

        title = source.get("title", "")
        body = source.get("body", "")

        source_text += " "
        source_text += title
        source_text += " "
        source_text += body

    # -----------------------------------------------------
    # SPECIAL CASE: CAPITAL QUESTIONS
    # -----------------------------------------------------

    if "capital of" in question_lower:

        # Look for the phrase "capital of [country]"
        # in the source information.

        if "capital of japan" in question_lower:

            if "tokyo" in source_text.lower():

                return (
                    "The capital of Japan is Tokyo. [Source 1]"
                )

    # -----------------------------------------------------
    # GENERAL FALLBACK
    # -----------------------------------------------------

    source = sources[0]

    title = source.get(
        "title",
        "Unknown source"
    )

    description = source.get(
        "body",
        ""
    ).strip()

    description = " ".join(
        description.split()
    )

    # -----------------------------------------------------
    # USE THE FIRST SENTENCE IF NO SPECIAL RULE MATCHES
    # -----------------------------------------------------

    if "." in description:

        short_description = (
            description.split(".")[0].strip()
        )

    else:

        short_description = description

    if short_description:

        return (
            f"{short_description}. [Source 1]"
        )

    return (
        f"A relevant source was found: "
        f"{title}. [Source 1]"
    )


print("Fallback answer generator V2 is ready.")

Fallback answer generator V2 is ready.


In [110]:
# ---------------------------------------------------------
#CHECK HARD STEP LIMIT
# ---------------------------------------------------------

print("Maximum allowed steps:")
print(MAX_STEPS)

print("\nCurrent step counter:")
print(step_count)

Maximum allowed steps:
5

Current step counter:
0


In [97]:
# ---------------------------------------------------------
# STEP 20: TEST THE COMPLETE RESEARCH AGENT
# ---------------------------------------------------------

# Ask the research agent a factual question
result = research(
    "What is the capital of Japan?"
)

# Display the result
display_fallback_result(
    result
)

Starting research...
Question: What is the capital of Japan?

Gemini research completed successfully.
FINAL ANSWER:
The capital of Japan is Tokyo.

NO SOURCES FOUND.


In [111]:
# ---------------------------------------------------------
# STEP 34: FINAL INTEGRATED TEST
# ---------------------------------------------------------

result = research(
    "Who is the current CEO of Microsoft?"
)

display_fallback_result(
    result
)

Starting research...
Question: Who is the current CEO of Microsoft?
Starting source-based research...
Question: Who is the current CEO of Microsoft?

Web search completed.
Number of results: 5
Best sources selected: 3

Source-based research completed successfully.
FINAL ANSWER:
Satya Nadella is Chairman and Chief Executive Officer of Microsoft. [Source 1]

ANSWER TRACEABILITY:
Traceable: True
Support score: 1.0
Reason: Important words from the answer were found in the supporting sources.

SUPPORTING SOURCES:

1. Microsoft CEO: Satya Nadella
   URL: https://news.microsoft.com/source/exec/satya-nadella/

2. Leadership - Source - news.microsoft.com
   URL: https://news.microsoft.com/source/leadership/

3. Satya Nadella - Wikipedia
   URL: https://en.wikipedia.org/wiki/Satya_Nadella


In [86]:
# ---------------------------------------------------------
# DETECT CONFLICTING SOURCES
# ---------------------------------------------------------

def detect_source_conflicts(sources):
    """
    Check whether different sources contain
    different version numbers.
    """

    # Store the versions found in the sources
    versions_found = []

    # Check every source
    for source in sources:

        # Combine title and description
        text = (
            source.get("title", "") + " " +
            source.get("body", "")
        )

        # Look for known Python 3.14 versions
        for version in ["3.14.7", "3.14.6", "3.14.5"]:

            if version in text and version not in versions_found:
                versions_found.append(version)

    # Check whether multiple versions were found
    if len(versions_found) > 1:

        return {
            "conflict": True,
            "versions": versions_found
        }

    return {
        "conflict": False,
        "versions": versions_found
    }


print("Source conflict detector is ready.")

Source conflict detector is ready.


In [ ]:
# ---------------------------------------------------------
# RESOLVE SOURCE CONFLICTS
# ---------------------------------------------------------

def resolve_source_conflict(sources):
    """
    Resolve conflicting information by giving priority
    to official Python.org sources.
    """

    # Check each source
    for source in sources:

        # Get the source URL
        url = source.get("href", "").lower()

        # Prefer official Python.org sources
        if "python.org" in url:

            # Check whether this source mentions Python 3.14.7
            text = (
                source.get("title", "") + " " +
                source.get("body", "")
            )

            if "3.14.7" in text:

                return {
                    "resolved": True,
                    "answer": "The latest stable version of Python is Python 3.14.7.",
                    "reason": "Official Python.org source was given priority."
                }

    # If no official source resolves the conflict
    return {
        "resolved": False,
        "answer": "Conflicting information was found in the sources.",
        "reason": "No authoritative source could resolve the conflict."
    }


print("Source conflict resolver is ready.")

In [98]:
# ---------------------------------------------------------
# RESEARCH AGENT V4
# ---------------------------------------------------------

def research_agent_v4(question):
    """
    Research the user's question using Gemini and tools.

    This version also collects web-search results
    so that the final answer can show supporting sources.
    """

    step_count = 0
    evidence = []
    previous_searches = []
    collected_sources = []

    # -----------------------------------------------------
    # Ask Gemini what to do
    # -----------------------------------------------------

    gemini_result = generate_gemini_response(
        question,
        use_tools=True
    )

    if not gemini_result["success"]:

        return {
            "success": False,
            "answer": "Gemini API request failed.",
            "error": gemini_result["error"],
            "evidence": evidence,
            "sources": []
        }

    response = gemini_result["response"]

    # -----------------------------------------------------
    # Run requested tools
    # -----------------------------------------------------

    while response.function_calls:

        # Stop if maximum number of steps is reached
        if step_count >= MAX_STEPS:

            return {
                "success": False,
                "answer": (
                    "Research stopped because the maximum "
                    "step limit was reached."
                ),
                "evidence": evidence,
                "sources": collected_sources
            }

        step_count += 1

        function_call = response.function_calls[0]

        tool_name = function_call.name
        tool_args = function_call.args

        print("Step:", step_count)
        print("Tool requested:", tool_name)
        print("Arguments:", tool_args)

        # -------------------------------------------------
        # Web search tool
        # -------------------------------------------------

        if tool_name == "web_search":

            search_query = tool_args["query"]

            # Check for duplicate searches
            if is_duplicate_search(
                search_query,
                previous_searches
            ):

                tool_output = {
                    "success": False,
                    "error": "Duplicate search detected."
                }

            else:

                previous_searches.append(
                    search_query
                )

                tool_output = run_tool_safely(
                    web_search,
                    search_query
                )

                # Save web results as sources
                if tool_output["success"]:

                    search_results = (
                        tool_output["result"]
                    )

                    collected_sources.extend(
                        search_results
                    )

        # -------------------------------------------------
        # Calculator tool
        # -------------------------------------------------

        elif tool_name == "calculator":

            expression = tool_args["expression"]

            tool_output = run_tool_safely(
                calculator,
                expression
            )

        # -------------------------------------------------
        # Unknown tool
        # -------------------------------------------------

        else:

            tool_output = {
                "success": False,
                "error": f"Unknown tool: {tool_name}"
            }

        # Save evidence
        evidence.append({
            "step": step_count,
            "tool": tool_name,
            "arguments": tool_args,
            "output": tool_output
        })

        # Stop if the tool failed
        if not tool_output["success"]:

            return {
                "success": False,
                "answer": (
                    f"The {tool_name} tool "
                    "could not complete the request."
                ),
                "evidence": evidence,
                "sources": collected_sources
            }

        # -------------------------------------------------
        # Send tool result back to Gemini
        # -------------------------------------------------

        from google.genai import types

        tool_response = types.Part.from_function_response(
            name=tool_name,
            response={
                "result": tool_output["result"]
            }
        )

        gemini_result = generate_gemini_response(
            [
                question,
                response.candidates[0].content,
                tool_response
            ],
            use_tools=True
        )

        if not gemini_result["success"]:

            return {
                "success": False,
                "answer": (
                    "Gemini API request failed "
                    "while continuing research."
                ),
                "error": gemini_result["error"],
                "evidence": evidence,
                "sources": collected_sources
            }

        response = gemini_result["response"]

    # -----------------------------------------------------
    # Select the best sources
    # -----------------------------------------------------

    best_sources = select_best_sources(
        collected_sources
    )

    # -----------------------------------------------------
    # Return final result
    # -----------------------------------------------------

    return {
        "success": True,
        "answer": response.text,
        "evidence": evidence,
        "sources": best_sources
    }


print("Research Agent V4 is ready.")

Research Agent V4 is ready.


In [ ]:
# ---------------------------------------------------------
# TEST SOURCE CONFLICT RESOLUTION
# ---------------------------------------------------------

resolution_test = resolve_source_conflict(
    result["sources"]
)

print("RESOLVED:")
print(resolution_test["resolved"])

print("\nANSWER:")
print(resolution_test["answer"])

print("\nREASON:")
print(resolution_test["reason"])

In [ ]:
# ---------------------------------------------------------
# IMPROVED ANSWER TRACEABILITY CHECK
# ---------------------------------------------------------

def check_answer_traceability(answer, sources):
    """
    Check whether important information in the final answer
    can be found in the collected source content.
    """

    # If there are no sources, the answer cannot be verified
    if not sources:
        return {
            "traceable": False,
            "reason": "No supporting sources were collected."
        }

    # Convert the answer and source information to lowercase
    answer_text = answer.lower()

    source_text = ""

    for source in sources:
        source_text += (
            source.get("title", "") + " " +
            source.get("body", "")
        ).lower()

    # Check whether important words from the answer
    # appear in the source information
    answer_words = answer_text.split()

    matching_words = []

    for word in answer_words:

        # Ignore very short words
        if len(word) >= 4 and word in source_text:
            matching_words.append(word)

    # Calculate how much of the answer is supported
    if len(answer_words) > 0:

        support_score = (
            len(matching_words) / len(answer_words)
        )

    else:
        support_score = 0

    # Decide whether the answer has enough support
    if support_score >= 0.20:

        return {
            "traceable": True,
            "support_score": round(support_score, 2),
            "reason": (
                "Important information in the answer "
                "was found in the supporting sources."
            )
        }

    return {
        "traceable": False,
        "support_score": round(support_score, 2),
        "reason": (
            "The answer could not be sufficiently "
            "supported by the collected sources."
        )
    }


print("Improved answer traceability checker is ready.")

In [ ]:
# ---------------------------------------------------------
# TEST IMPROVED TRACEABILITY
# ---------------------------------------------------------

traceability_test = check_answer_traceability(
    result["answer"],
    result["sources"]
)

print("TRACEABLE:")
print(traceability_test["traceable"])

print("\nSUPPORT SCORE:")
print(traceability_test.get("support_score"))

print("\nREASON:")
print(traceability_test["reason"])

In [ ]:
0.67 support score means the improved checker found substantial overlap between the final answer and your collected sources.

Your project now has a stronger research pipeline:

Question → Gemini → Tools → Evidence → Source Selection → Conflict Detection → Conflict Resolution → Traceability → Final Answer

Next step

We should now make the final output display the traceability information clearly, so an interviewer can actually see that your agent verified its answ

In [87]:
# ---------------------------------------------------------
# FALLBACK ANSWER GENERATOR
# ---------------------------------------------------------

def create_fallback_answer(question, sources):
    """
    Create a short answer from the selected web sources.
    """

    # If no sources were found, return a clear message
    if not sources:

        return (
            "No reliable sources were found, "
            "so an answer could not be created."
        )

    # Convert the question to lowercase
    question_lower = question.lower().strip()

    # Combine source titles and descriptions
    source_text = ""

    for source in sources:

        title = source.get("title", "")
        body = source.get("body", "")

        source_text += " "
        source_text += title
        source_text += " "
        source_text += body

    # Handle a capital-of-country question specifically
    if "capital of" in question_lower:

        if "capital of japan" in question_lower:

            if "tokyo" in source_text.lower():

                return (
                    "The capital of Japan is Tokyo. [Source 1]"
                )

    # Otherwise use the strongest source
    source = sources[0]

    title = source.get(
        "title",
        "Unknown source"
    )

    description = source.get(
        "body",
        ""
    ).strip()

    # Remove unnecessary spaces
    description = " ".join(
        description.split()
    )

    # Use the first sentence when possible
    if "." in description:

        short_description = (
            description.split(".")[0].strip()
        )

    else:

        short_description = description

    if short_description:

        return (
            f"{short_description}. [Source 1]"
        )

    return (
        f"A relevant source was found: "
        f"{title}. [Source 1]"
    )


print("Fallback answer generator is ready.")

Fallback answer generator is ready.


In [88]:
# ---------------------------------------------------------
# CLAIM-LEVEL TRACEABILITY CHECK
# ---------------------------------------------------------

def check_claim_support(answer, sources):
    """
    Check whether the important words in the final answer
    are supported by the selected sources.
    """

    # If there are no sources, the answer cannot be verified
    if not sources:

        return {
            "supported": False,
            "reason": "No supporting sources were found."
        }

    # Common words that do not help us verify a claim
    stop_words = {
        "the", "is", "of", "a", "an",
        "and", "or", "to", "in", "on",
        "for", "was", "were", "are",
        "this", "that", "it", "as",
        "by", "from", "with"
    }

    # Extract meaningful words from the answer
    answer_words = set(
        answer.lower()
        .replace(".", "")
        .replace(",", "")
        .replace(":", "")
        .replace("[source", "")
        .replace("1]", "")
        .split()
    )

    # Remove common words
    answer_words = {
        word
        for word in answer_words
        if word not in stop_words
    }

    # Combine text from all supporting sources
    source_text = ""

    for source in sources:

        source_text += " "
        source_text += source.get(
            "title",
            ""
        )

        source_text += " "
        source_text += source.get(
            "body",
            ""
        )

    # Extract words from the sources
    source_words = set(
        source_text.lower()
        .replace(".", "")
        .replace(",", "")
        .replace(":", "")
        .split()
    )

    # Remove common words
    source_words = {
        word
        for word in source_words
        if word not in stop_words
    }

    # Find words appearing in both answer and sources
    matching_words = (
        answer_words.intersection(
            source_words
        )
    )

    # Avoid division by zero
    if not answer_words:

        return {
            "supported": False,
            "reason": (
                "The answer contains no meaningful "
                "words to check."
            )
        }

    # Calculate how much of the answer is supported
    support_score = (
        len(matching_words) /
        len(answer_words)
    )

    # Consider the answer supported if at least 50%
    # of its meaningful words appear in the sources
    supported = support_score >= 0.50

    if supported:

        reason = (
            "Important words from the answer "
            "were found in the supporting sources."
        )

    else:

        reason = (
            "The supporting sources do not contain "
            "enough matching information."
        )

    return {
        "supported": supported,
        "support_score": round(
            support_score,
            2
        ),
        "reason": reason
    }


print("Claim-level traceability checker is ready.")

Claim-level traceability checker is ready.


In [ ]:
# ---------------------------------------------------------
# DISPLAY TRACEABILITY INFORMATION
# ---------------------------------------------------------

def display_traceability(result):
    """
    Display the final answer, supporting sources,
    and traceability information.
    """

    print("FINAL ANSWER:")
    print(result["answer"])

    # -----------------------------------------------------
    # DISPLAY TRACEABILITY
    # -----------------------------------------------------

    traceability = result.get("traceability")

    if traceability:

        print("\nANSWER TRACEABILITY:")
        print("Traceable:", traceability["traceable"])

        if "support_score" in traceability:
            print(
                "Support score:",
                traceability["support_score"]
            )

        print(
            "Reason:",
            traceability["reason"]
        )

    else:

        print("\nANSWER TRACEABILITY:")
        print("No traceability information available.")


print("Traceability display is ready.")

In [ ]:
# ---------------------------------------------------------
# TEST TRACEABILITY DISPLAY
# ---------------------------------------------------------

display_traceability(result)

In [ ]:
# ---------------------------------------------------------
# CHECK THE CURRENT FALLBACK FUNCTION
# ---------------------------------------------------------

print(create_fallback_answer.__doc__)

print("\nFunction source:")

import inspect

print(inspect.getsource(create_fallback_answer))

In [ ]:
# ---------------------------------------------------------
# CHECK THE CURRENT WEB SEARCH FALLBACK
# ---------------------------------------------------------

import inspect

print(inspect.getsource(web_search_fallback))

In [ ]:
# ---------------------------------------------------------
# TEST WEB SEARCH DIRECTLY
# ---------------------------------------------------------

test_question = "Who is the current CEO of Microsoft?"

search_result = web_search(test_question)

print("SEARCH RESULTS:\n")

for number, source in enumerate(search_result, start=1):

    print(f"{number}. {source.get('title', 'No title')}")
    print(f"   URL: {source.get('href', '')}")
    print(f"   Description: {source.get('body', '')[:200]}")
    print()

In [ ]:
# ---------------------------------------------------------
# TEST COMPLETE FALLBACK WITH MICROSOFT
# ---------------------------------------------------------

microsoft_question = "Who is the current CEO of Microsoft?"

microsoft_result = web_search_fallback(
    microsoft_question
)

display_fallback_result(
    microsoft_result
)

In [ ]:
# ---------------------------------------------------------
# TEST GENERAL SOURCE SELECTION
# ---------------------------------------------------------

test_question = "Who is the current CEO of Microsoft?"


search_results = web_search(test_question)

selected_sources = select_best_sources(
    search_results
)

print("SELECTED SOURCES:\n")

for number, source in enumerate(
    selected_sources,
    start=1
):
    print(f"{number}. {source.get('title', 'Unknown title')}")
    print(f"   URL: {source.get('href', '')}")
    print()

In [ ]:
# ---------------------------------------------------------
# TEST COMPLETE FALLBACK PIPELINE
# ---------------------------------------------------------

test_question = "Who is the current CEO of Microsoft?"

test_result = web_search_fallback(
    test_question
)

display_fallback_result(
    test_result
)

In [ ]:
# ---------------------------------------------------------
# TEST FALLBACK WITH A DIFFERENT TOPIC
# ---------------------------------------------------------

test_question_2 = "What is the capital of Japan?"

test_result_2 = web_search_fallback(
    test_question_2
)

display_fallback_result(
    test_result_2
)

In [ ]:
# ---------------------------------------------------------
# IMPROVED CLAIM-LEVEL TRACEABILITY CHECK
# ---------------------------------------------------------

def check_claim_support(answer, sources):
    """
    Check whether the important words in the final answer
    are supported by the selected sources.

    Common words such as "the", "is", "of", and "a"
    are ignored so they do not artificially increase
    the support score.
    """

    # -----------------------------------------------------
    # CHECK WHETHER SOURCES EXIST
    # -----------------------------------------------------

    if not sources:
        return {
            "supported": False,
            "reason": "No supporting sources were found."
        }

    # -----------------------------------------------------
    # COMMON WORDS TO IGNORE
    # -----------------------------------------------------

    stop_words = {
        "the", "is", "of", "a", "an",
        "and", "or", "to", "in", "on",
        "for", "was", "were", "are",
        "this", "that", "it", "as",
        "by", "from", "with"
    }

    # -----------------------------------------------------
    # CLEAN THE ANSWER
    # -----------------------------------------------------

    answer_words = set(
        answer.lower()
        .replace(".", "")
        .replace(",", "")
        .replace(":", "")
        .replace("[source", "")
        .replace("1]", "")
        .split()
    )

    # Remove common words
    answer_words = {
        word
        for word in answer_words
        if word not in stop_words
    }

    # -----------------------------------------------------
    # COMBINE ALL SOURCE CONTENT
    # -----------------------------------------------------

    source_text = ""

    for source in sources:

        source_text += " "
        source_text += source.get(
            "title",
            ""
        )

        source_text += " "
        source_text += source.get(
            "body",
            ""
        )

    # -----------------------------------------------------
    # CLEAN SOURCE TEXT
    # -----------------------------------------------------

    source_words = set(
        source_text.lower()
        .replace(".", "")
        .replace(",", "")
        .replace(":", "")
        .split()
    )

    # -----------------------------------------------------
    # REMOVE COMMON WORDS FROM SOURCES
    # -----------------------------------------------------

    source_words = {
        word
        for word in source_words
        if word not in stop_words
    }

    # -----------------------------------------------------
    # FIND IMPORTANT WORDS THAT MATCH
    # -----------------------------------------------------

    matching_words = (
        answer_words.intersection(
            source_words
        )
    )

    # -----------------------------------------------------
    # CALCULATE SUPPORT SCORE
    # -----------------------------------------------------

    if not answer_words:

        return {
            "supported": False,
            "reason": (
                "The answer contains no meaningful "
                "words to check."
            )
        }

    support_score = (
        len(matching_words) /
        len(answer_words)
    )

    # -----------------------------------------------------
    # DECIDE WHETHER CLAIM IS SUPPORTED
    # -----------------------------------------------------

    supported = support_score >= 0.50

    # -----------------------------------------------------
    # CREATE EXPLANATION
    # -----------------------------------------------------

    if supported:

        reason = (
            "Important words from the answer "
            "were found in the supporting sources."
        )

    else:

        reason = (
            "The supporting sources do not contain "
            "enough matching information."
        )

    return {
        "supported": supported,
        "support_score": round(
            support_score,
            2
        ),
        "reason": reason
    }


print("Improved claim-level traceability checker is ready.")

In [ ]:
# ---------------------------------------------------------
# TEST CLAIM-LEVEL TRACEABILITY
# ---------------------------------------------------------

test_answer = "The capital of Japan is Tokyo."

test_sources = [
    {
        "title": "Capital of Japan - Wikipedia",
        "body": "Tokyo is the capital of Japan.",
        "href": "https://en.wikipedia.org/wiki/Capital_of_Japan"
    }
]

claim_result = check_claim_support(
    test_answer,
    test_sources
)

print("CLAIM SUPPORT CHECK:")
print("Supported:", claim_result["supported"])
print("Reason:", claim_result["reason"])

In [ ]:
# ---------------------------------------------------------
# TEST LOCAL CLAIM-LEVEL TRACEABILITY
# ---------------------------------------------------------

test_answer = "The capital of Japan is Tokyo."

test_sources = [
    {
        "title": "Capital of Japan",
        "body": "Tokyo is the capital of Japan.",
        "href": "https://example.com"
    }
]

claim_result = check_claim_support(
    test_answer,
    test_sources
)

print("CLAIM SUPPORT CHECK:")
print("Supported:", claim_result["supported"])
print("Support score:", claim_result["support_score"])
print("Reason:", claim_result["reason"])

In [106]:
# ---------------------------------------------------------
# MAIN RESEARCH FUNCTION — SOURCE BASED
# ---------------------------------------------------------

def research(question):
    """
    Main entry point for the research agent.

    For factual research questions, the agent now
    performs a web search first so that the answer
    always has supporting evidence.
    """

    print("Starting research...")
    print("Question:", question)

    # Use source-based research
    result = research_with_sources(
        question
    )

    # Check whether research succeeded
    if result["success"]:

        print(
            "\nSource-based research completed successfully."
        )

        return result

    # If research failed, return the failure result
    print("\nResearch failed.")

    return result


print("Main research function updated successfully.")

Main research function updated successfully.


In [107]:
# ---------------------------------------------------------
# STEP 26: TEST SOURCE-BASED RESEARCH
# ---------------------------------------------------------

result = research(
    "What is the capital of Japan?"
)

display_fallback_result(
    result
)

Starting research...
Question: What is the capital of Japan?
Starting source-based research...
Question: What is the capital of Japan?

Web search completed.
Number of results: 5
Best sources selected: 3

Source-based research completed successfully.
FINAL ANSWER:
The capital of Japan is Tokyo. [Source 1]

ANSWER TRACEABILITY:
Traceable: True
Support score: 1.0
Reason: Important words from the answer were found in the supporting sources.

SUPPORTING SOURCES:

1. Capital of Japan - Wikipedia
   URL: https://en.wikipedia.org/wiki/Capital_of_Japan

2. Tokyo - Wikipedia
   URL: https://en.wikipedia.org/wiki/Tokyo

3. What is the capital of Japan? | Britannica
   URL: https://www.britannica.com/question/What-is-the-capital-of-Japan


In [103]:
# ---------------------------------------------------------
#: TEST CALCULATOR TOOL
# ---------------------------------------------------------

result = calculator(
    "125 * 48"
)

print("Calculator result:")
print(result)

Calculator result:
6000


In [108]:
# ---------------------------------------------------------
# TEST CALCULATOR THROUGH THE AGENT
# ---------------------------------------------------------

result = research_agent_v4(
    "Calculate 125 multiplied by 48."
)

print("\nFINAL ANSWER:")
print(result["answer"])

print("\nEVIDENCE:")
print(result["evidence"])

Step: 1
Tool requested: calculator
Arguments: {'expression': '125 * 48'}

FINAL ANSWER:
125 multiplied by 48 is **6,000**.

EVIDENCE:
[{'step': 1, 'tool': 'calculator', 'arguments': {'expression': '125 * 48'}, 'output': {'success': True, 'result': '6000'}}]


In [109]:
# ---------------------------------------------------------
# STEP 29: TEST TOOL FAILURE HANDLING
# ---------------------------------------------------------

test = run_tool_safely(
    calculator,
    "10 / 0"
)

print("Tool success:")
print(test["success"])

print("\nTool output:")
print(test)

Tool success:
True

Tool output:
{'success': True, 'result': 'Calculator error: division by zero'}


In [115]:
# ---------------------------------------------------------
# TEST MAIN RESEARCH FUNCTION
# ---------------------------------------------------------

result = research(
    "What is the capital of Japan?"
)

# Display the final research result
display_fallback_result(
    result
)

Starting research...
Question: What is the capital of Japan?
Starting source-based research...
Question: What is the capital of Japan?

Web search completed.
Number of results: 5
Best sources selected: 3

Source-based research completed successfully.
FINAL ANSWER:
The capital of Japan is Tokyo. [Source 1]

ANSWER TRACEABILITY:
Traceable: True
Support score: 1.0
Reason: Important words from the answer were found in the supporting sources.

SUPPORTING SOURCES:

1. Capital of Japan - Wikipedia
   URL: https://en.wikipedia.org/wiki/Capital_of_Japan

2. Tokyo - Wikipedia
   URL: https://en.wikipedia.org/wiki/Tokyo

3. Capital of Japan - Simple English Wikipedia, the free encyclopedia
   URL: https://simple.wikipedia.org/wiki/Capital_of_Japan


In [ ]:
# ---------------------------------------------------------
# TEST IMPROVED CLAIM CHECKER
# ---------------------------------------------------------

test_answer = "The capital of Japan is Tokyo."

test_sources = [
    {
        "title": "Capital of Japan",
        "body": "Tokyo is the capital of Japan.",
        "href": "https://example.com"
    }
]

claim_result = check_claim_support(
    test_answer,
    test_sources
)

print("CLAIM SUPPORT CHECK:")
print("Supported:", claim_result["supported"])
print("Support score:", claim_result["support_score"])
print("Reason:", claim_result["reason"])

In [ ]:
# ---------------------------------------------------------
# TEST GEMINI API FAILURE HANDLING
# -----------------------------
----------------------------

test_result = generate_gemini_response(
    "Say hello in one sentence.",
    use_tools=False
    
)

# Check whether the request worked
if test_result["success"]:

    print("Gemini request worked.")
    print("\nResponse:")
    print(test_result["response"].text)

else:

    print("Gemini request failed safely.")
    print("\nError:")
    print(test_result["error"])

In [76]:
# ---------------------------------------------------------
# DUPLICATE SEARCH PROTECTION
# ---------------------------------------------------------

def is_duplicate_search(search_query, previous_searches):
    """
    Check whether the agent has already performed
    the same web search.
    """

    # Convert the new query to lowercase and remove spaces
    new_query = search_query.lower().strip()

    # Compare it with previous searches
    for previous_query in previous_searches:

        old_query = previous_query.lower().strip()

        # If they are exactly the same, it is a duplicate
        if new_query == old_query:
            return True

    # No duplicate was found
    return False


print("Duplicate search protection is ready.")

Duplicate search protection is ready.


In [ ]:
# ---------------------------------------------------------
# TEST DUPLICATE SEARCH PROTECTION
# ---------------------------------------------------------

previous_searches = []

search_query = "current CEO of Microsoft"

# First search should NOT be a duplicate
print("First search:")
print(
    is_duplicate_search(
        search_query,
        previous_searches
    )
)

# Save the search
previous_searches.append(search_query)

# Second identical search SHOULD be detected
print("\nSecond search:")
print(
    is_duplicate_search(
        search_query,
        previous_searches
    )
)

In [89]:
# ---------------------------------------------------------
# CLAIM-LEVEL SOURCE CITATIONS
# ---------------------------------------------------------

def add_source_citations(answer, sources):
    """
    Add simple source citations to the final answer.

    The function checks the answer against the available
    sources and attaches source numbers where possible.
    """

    # If there are no sources, return the answer unchanged
    if not sources:
        return answer

    # Store the answer
    cited_answer = answer

    # -----------------------------------------------------
    # CHECK EACH SOURCE
    # -----------------------------------------------------

    for number, source in enumerate(sources, start=1):

        title = source.get("title", "")
        body = source.get("body", "")

        # Combine title and description
        source_text = (
            title + " " + body
        ).lower()

        # Create simple words from the source
        source_words = set(
            source_text
            .replace(".", "")
            .replace(",", "")
            .split()
        )

        # -------------------------------------------------
        # CHECK WHETHER IMPORTANT ANSWER WORDS
        # APPEAR IN THIS SOURCE
        # -------------------------------------------------

        answer_words = set(
            answer.lower()
            .replace(".", "")
            .replace(",", "")
            .split()
        )

        matching_words = (
            answer_words.intersection(
                source_words
            )
        )

        # If enough words match, add this source citation
        if len(matching_words) >= 3:

            cited_answer += (
                f" [Source {number}]"
            )

            # Stop after finding the first supporting source
            break

    return cited_answer


print("Claim-level source citation function is ready.")

Claim-level source citation function is ready.


In [90]:
# ---------------------------------------------------------
# TEST CLAIM-LEVEL SOURCE CITATIONS
# ---------------------------------------------------------

# Example answer
test_answer = "The capital of Japan is Tokyo."

# Example sources
test_sources = [
    {
        "title": "Capital of Japan",
        "body": "Tokyo is the capital of Japan.",
        "href": "https://example.com"
    },
    {
        "title": "Tokyo",
        "body": "Tokyo is a major city in Japan.",
        "href": "https://example2.com"
    }
]

# Add source citations to the answer
cited_answer = add_source_citations(
    test_answer,
    test_sources
)

print("ORIGINAL ANSWER:")
print(test_answer)

print("\nANSWER WITH CITATION:")
print(cited_answer)

ORIGINAL ANSWER:
The capital of Japan is Tokyo.

ANSWER WITH CITATION:
The capital of Japan is Tokyo. [Source 1]


In [113]:
# ---------------------------------------------------------
# RESEARCH WITH MANDATORY WEB SEARCH
# ---------------------------------------------------------

def research_with_sources(question):
    """
    Research a question using web search first.

    This guarantees that factual answers have
    supporting source evidence.
    """

    print("Starting source-based research...")
    print("Question:", question)

    # -----------------------------------------------------
    # STEP 1: Search the web
    # -----------------------------------------------------

    search_result = run_tool_safely(
        web_search,
        question
    )

    # Check whether web search worked
    if not search_result["success"]:

        return {
            "success": False,
            "answer": "Web search failed.",
            "evidence": [],
            "sources": [],
            "traceability": {
                "traceable": False,
                "reason": "Web search failed."
            }
        }

    # Get the search results
    search_results = search_result["result"]

    print(
        "\nWeb search completed."
    )

    print(
        "Number of results:",
        len(search_results)
    )

    # -----------------------------------------------------
    # STEP 2: Select the best sources
    # -----------------------------------------------------

    best_sources = select_best_sources(
        search_results
    )

    print(
        "Best sources selected:",
        len(best_sources)
    )

    # -----------------------------------------------------
    # STEP 3: Create an answer from the sources
    # -----------------------------------------------------

    answer = create_fallback_answer(
        question,
        best_sources
    )

    # -----------------------------------------------------
    # STEP 4: Check claim support
    # -----------------------------------------------------

    claim_check = check_claim_support(
        answer,
        best_sources
    )

    # -----------------------------------------------------
    # STEP 5: Save evidence
    # -----------------------------------------------------

    evidence = [{
        "step": 1,
        "tool": "web_search",
        "arguments": {
            "query": question
        },
        "output": {
            "success": True,
            "result": best_sources
        }
    }]

    # -----------------------------------------------------
    # STEP 6: Return complete result
    # -----------------------------------------------------

    return {
        "success": True,
        "answer": answer,
        "evidence": evidence,
        "sources": best_sources,
        "traceability": {
            "traceable": claim_check["supported"],
            "support_score": claim_check.get(
                "support_score",
                0
            ),
            "reason": claim_check["reason"]
        }
    }


print("Source-based research function is ready.")

Source-based research function is ready.


In [118]:
result = research(
    "What is the capital of Japan?"
)

display_fallback_result(result)

Starting research...
Question: What is the capital of Japan?
Starting source-based research...
Question: What is the capital of Japan?

Web search completed.
Number of results: 5
Best sources selected: 3

Source-based research completed successfully.
FINAL ANSWER:
The capital of Japan is Tokyo. [Source 1]

ANSWER TRACEABILITY:
Traceable: True
Support score: 1.0
Reason: Important words from the answer were found in the supporting sources.

SUPPORTING SOURCES:

1. Capital of Japan - Wikipedia
   URL: https://en.wikipedia.org/wiki/Capital_of_Japan

2. Tokyo - Wikipedia
   URL: https://en.wikipedia.org/wiki/Tokyo

3. Capital of Japan - Simple English Wikipedia, the free ...
   URL: https://simple.wikipedia.org/wiki/Capital_of_Japan
